###  Insight Estratégico — Concentração Geográfica da Receita

* **Diagnóstico:** Alta densidade de faturamento concentrada no eixo Sudeste e Sul (especialmente nas regiões metropolitanas de São Paulo e Rio de Janeiro).
* **Oportunidade:** Viabilidade técnica para abertura de Hubs de Distribuição Avançados (*Dark Stores*) nessas praças para reduzir o custo do frete e o tempo de entrega.
* **Recomendação:** Direcionar campanhas de marketing para capitais com alto volume e bom ticket médio fora do eixo principal, reduzindo a dependência de praças saturadas.

In [27]:
import os
import pandas as pd
import plotly.express as px

# 1. Carrega os Data Marts em Parquet
df_pedidos = pd.read_parquet("data/mart_pedidos_performance.parquet")
df_rfm = pd.read_parquet("data/mart_rfm_clientes.parquet")

# 2. Configura a exibição do Plotly 
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [22]:
import pandas as pd
import plotly.express as px

# 1. Filtra registros geográficos válidos do Data Mart de RFM
df_geo_valid = df_rfm.dropna(subset=['latitude', 'longitude'])

# 2. Mapa de Calor de Densidade Geográfica de Receita
fig_mapa = px.density_map(
    df_geo_valid,
    lat='latitude',
    lon='longitude',
    z='ValorTotal_R$',
    radius=8,
    center=dict(lat=-14.2350, lon=-51.9253),
    zoom=3,
    map_style="open-street-map",
    title="Concentração Geográfica da Receita por Cliente (R$)",
    color_continuous_scale='Viridis'
)

fig_mapa.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0}
)

fig_mapa.show()

# 3. Atualiza a cópia em HTML na pasta assets
fig_mapa.write_html("assets/08_mapa_densidade_receita.html")

###  Insight Estratégico — Tendência Temporal e Diversificação Regional

* **Diagnóstico de Risco:** Alta dependência e concentração de receita na Região Sudeste, que responde pela maior parte do volume e puxa o pico isolado de vendas em Novembro/2017 (efeito Black Friday).
* **Oportunidade de Mercado:** As regiões com menor representatividade ou crescimento lento (como Norte e Centro-Oeste) representam mercados subexplorados, onde a barreira do frete/prazo freia a conversão.
* **Recomendação Estratégica:** 
  1. **Diversificação da Base:** Investir em campanhas de aquisição direcionadas e incentivos de frete para regiões fora do eixo Sudeste/Sul para reduzir a vulnerabilidade do negócio.
  2. **Eficiência Logística:** Planejar a descentralização de estoques para polos regionais antes de grandes datas comemorativas, aumentando a conversão nesses mercados alternativos.

In [23]:

# 1. Carrega o Data Mart de Pedidos
df_pedidos = pd.read_parquet("data/mart_pedidos_performance.parquet")

# 2. Mapeamento de Estados para Macroregiões
depara_regiao = {
    'AC': 'Norte', 'AL': 'Nordeste', 'AP': 'Norte', 'AM': 'Norte', 'BA': 'Nordeste',
    'CE': 'Nordeste', 'DF': 'Centro-Oeste', 'ES': 'Sudeste', 'GO': 'Centro-Oeste',
    'MA': 'Nordeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'MG': 'Sudeste',
    'PA': 'Norte', 'PB': 'Nordeste', 'PR': 'Sul', 'PE': 'Nordeste', 'PI': 'Nordeste',
    'RJ': 'Sudeste', 'RN': 'Nordeste', 'RS': 'Sul', 'RO': 'Norte', 'RR': 'Norte',
    'SC': 'Sul', 'SP': 'Sudeste', 'SE': 'Nordeste', 'TO': 'Norte'
}

# 3. Tratamento e Agregação Temporal por Região
df_pedidos['Regiao'] = df_pedidos['customer_state'].map(depara_regiao)
df_pedidos['Ano_Mes'] = df_pedidos['order_purchase_timestamp'].dt.to_period('M').astype(str)

df_evolucao_regiao = (
    df_pedidos.groupby(['Ano_Mes', 'Regiao'])['order_id']
    .nunique()
    .reset_index()
    .rename(columns={'order_id': 'Total_Pedidos'})
)

# Filtra intervalo de datas estáveis (2017 a meados de 2018)
df_evolucao_regiao = df_evolucao_regiao[
    (df_evolucao_regiao['Ano_Mes'] >= '2017-01') & (df_evolucao_regiao['Ano_Mes'] <= '2018-08')
]

# 4. Construção do Gráfico de Linhas Interativo
fig_linha_regiao = px.line(
    df_evolucao_regiao,
    x='Ano_Mes',
    y='Total_Pedidos',
    color='Regiao',
    markers=True,
    title="Evolução Mensal do Volume de Pedidos por Região",
    labels={
        'Ano_Mes': 'Mês de Compra',
        'Total_Pedidos': 'Quantidade de Pedidos',
        'Regiao': 'Região'
    },
    color_discrete_sequence=px.colors.qualitative.Set1
)

fig_linha_regiao.update_layout(
    hovermode="x unified",
    xaxis_tickangle=-45,
    margin=dict(t=60, b=60, l=40, r=40)
)

fig_linha_regiao.show()

# 5. Salva na pasta assets
fig_linha_regiao.write_html("assets/11_evolucao_pedidos_regiao.html")

###  Insight Estratégico — Segmentação de Base (RFM)

* **Diagnóstico:** Apenas **6.4%** da base de clientes enquadra-se no perfil **Recente / Valioso** (VIPs), enquanto **24.4%** já apresentam sinais claros de **Em Risco / Churn**.
* **Gargalo:** O e-commerce possui uma base fortemente caracterizada por compras pontuais e isoladas, demonstrando baixa retenção orgânica.
* **Recomendação:** Criar réguas automatizadas de CRM para a base "Em Risco", oferecendo cupons de recompra dentro da janela crítica de 60 a 90 dias após o pedido inicial.

####  Citério de Classificação da Matriz RFM

A segmentação dos clientes foi calculada com base na **Recência (R)** e no **Valor Monetário (M)**:

* **Recente / Valioso:** Clientes no topo de Recência (R=4) e Valor (M=4).
* **Em Risco / Churn:** Clientes de alto valor acumulado (M≥3), mas sem compras recentes (R≤2).
* **Cliente Ativo / Recente:** Clientes com compras recentes (R≥3) ainda em fase de engajamento.
* **Atenção Necessária:** Clientes com baixa recência e baixo valor acumulado.

In [24]:
# Célula — Distribuição dos Segmentos RFM (Formatado)
df_seg_count = df_rfm['Segmento_Cliente'].value_counts().reset_index()
df_seg_count.columns = ['Segmento', 'Quantidade']

fig_rfm = px.pie(
    df_seg_count,
    names='Segmento',
    values='Quantidade',
    hole=0.5,
    title="Distribuição Percentual dos Segmentos de Clientes (RFM)",
    color_discrete_sequence=px.colors.qualitative.Set2
)

# Ajuste fino da formatação visual
fig_rfm.update_traces(
    textposition='outside',          
    textinfo='percent+label',        
    hovertemplate='<b>%{label}</b><br>Clientes: %{value:,.0f}<br>Percentual: %{percent}<extra></extra>'
)

fig_rfm.update_layout(
    showlegend=True,
    legend_title_text="Segmentos",
    margin=dict(t=60, b=40, l=40, r=40)
)

fig_rfm.show()

###  Insight Estratégico — Monetização por Perfil de Cliente

* **Diagnóstico:** O grupo **Cliente Ativo / Recente** lidera o volume financeiro absoluto devido ao grande número de compradores, mas o ticket médio unitário do segmento **Recente / Valioso** é substancialmente superior.
* **Oportunidade:** Existe uma grande margem de crescimento para trabalhar o *LTV (Lifetime Value)* da base intermediária.
* **Recomendação:** Estruturar programas de fidelidade (*Loyalty*) e ofertas de *cross-selling* focados em migrar "Clientes Ativos" para o perfil "Recente / Valioso".

In [25]:
df_seg_val = df_rfm.groupby('Segmento_Cliente')['ValorTotal_R$'].sum().reset_index()
df_seg_val = df_seg_val.sort_values(by='ValorTotal_R$', ascending=True)

fig_receita_seg = px.bar(
    df_seg_val,
    x='ValorTotal_R$',
    y='Segmento_Cliente',
    orientation='h',
    text_auto='.2s',
    title="Receita Acumulada (R$) por Segmento de Cliente",
    labels={'ValorTotal_R$': 'Receita Total (R$)', 'Segmento_Cliente': 'Segmento'},
    color='ValorTotal_R$',
    color_continuous_scale='Viridis'
)
fig_receita_seg.show()

###  Insight Estratégico — Eficiência e Gargalos Logísticos (SLA)

* **Diagnóstico:** Disparidade expressiva no tempo de entrega entre o Norte/Nordeste (ultrapassando 20 a 25 dias em estados como RR, AP e AM) versus Sudeste/Sul (SP e PR abaixo de 10 dias).
* **Risco de Negócio:** O elevado tempo de espera é o principal fator gerador de avaliações negativas (*reviews* baixos), cancelamentos e pedidos de reembolso.
* **Recomendação:** Reavaliar parcerias de frete rodoviário/aéreo para o Norte/Nordeste e calibrar as estimativas de SLA exibidas no checkout para alinhar a expectativa do consumidor.

In [26]:
# Célula 4 — Lead Time Médio de Entrega por Estado (UF)
df_lead_uf = df_pedidos.groupby('customer_state')['dias_entrega_real'].mean().reset_index()
df_lead_uf['dias_entrega_real'] = df_lead_uf['dias_entrega_real'].round(1)
df_lead_uf = df_lead_uf.sort_values(by='dias_entrega_real', ascending=False)

fig_lead = px.bar(
    df_lead_uf,
    x='customer_state',
    y='dias_entrega_real',
    title="Tempo Médio de Entrega (Lead Time em Dias) por Estado (UF)",
    labels={'customer_state': 'Estado (UF)', 'dias_entrega_real': 'Média de Dias para Entrega'},
    text='dias_entrega_real',
    color='dias_entrega_real',
    color_continuous_scale='Blues'
)
fig_lead.update_traces(textposition='outside')
fig_lead.show()